In [2]:
from joblib.externals.loky import reusable_executor

from Util.Problems import Problem, solution

import math
import Util.math_functions as mathf

class P029(Problem):
    number = 29
    title = "Distinct Powers"
    description = """<p>Consider all integer combinations of $a^b$ for $2 \\le a \\le 5$ and $2 \\le b \\le 5$:
$$\\begin{array}{rrrr}
2^2=4, &2^3=8, &2^4=16, &2^5=32\\\\
3^2=9, &3^3=27, &3^4=81, &3^5=243\\\\
4^2=16, &4^3=64, &4^4=256, &4^5=1024\\\\
\\end{array}$$</p><p>If they are then placed in numerical order, with any repeats removed, we get the following sequence of $15$ distinct terms:
$$4, 8, 9, 16, 25, 27, 32, 64, 81, 125, 243, 256, 625, 1024, 3125.$$</p><p>How many distinct terms are in the sequence generated by $a^b$ for $2 \\le a \\le 100$ and $2 \\le b \\le 100$?</p>"""
    upper_limit = 100

In [3]:
p = P029()
p.describe()

## Problem 29: Distinct Powers

<p>Consider all integer combinations of $a^b$ for $2 \le a \le 5$ and $2 \le b \le 5$:
$$\begin{array}{rrrr}
2^2=4, &2^3=8, &2^4=16, &2^5=32\\
3^2=9, &3^3=27, &3^4=81, &3^5=243\\
4^2=16, &4^3=64, &4^4=256, &4^5=1024\\
\end{array}$$</p><p>If they are then placed in numerical order, with any repeats removed, we get the following sequence of $15$ distinct terms:
$$4, 8, 9, 16, 25, 27, 32, 64, 81, 125, 243, 256, 625, 1024, 3125.$$</p><p>How many distinct terms are in the sequence generated by $a^b$ for $2 \le a \le 100$ and $2 \le b \le 100$?</p>

### Solution notes
For any $a$ holds: if $a$ is prime, all values of b provide a unique result. The same goes if $a$ can not be written in the form $x^n$. When $a$ is of the form $x^n$, however, some of the values for $b$ provide results which were already discovered when $a$ was equal to $x, x^2 ...$ or $x^{n-1}$. These results can be filtered out as follows:

If $a$ can be written as $x^n$, the values it will generate, will be those of $(x^n)^b$. Therefore, if $n\times b < upper\_limit$ the values have already been calculated when calculating $x$. Furthermore, a range of values of $(x^n)^b$ will already have been calculated by $x^2, x^3 ... x^(n-1)$.

We will loop through all values for $b$ to determine if $(x^n)^b$ has been calculated before. Assuming $n < upper\_limit$, all $b < n$ have been calculated before at $(x^b)^n$. For each remaining value of b, if it is prime this value has not been calculated before. If it is composite, loop through previous powers of x $2, 3 ... (n-1)$. If $b$ is divisible by such a power, this power has calculated the value iff $\frac{b}{previous\_power} \times n \le upper\_limit $

In [210]:
@solution(P029, first=True, make_fast=False, warmup_args=(P029.upper_limit, ))
def counting_solutions(upper_limit):
    def get_single_factor(n):
        root = 0
        for i in range(int(math.log2(n)), 1, -1):
            integer_root = int(n ** (1/i))
            if integer_root ** i == n:
                root = integer_root
                break
        return root

    def smallest_common_factor(n, m):
        if n == 1 or m == 1:
            return n * m
        if mathf.is_prime(n) or mathf.is_prime(m):
            if m % n == 0:
                return n
            elif n % m == 0:
                return m
            return n * m
        i = 2
        while not (n % i == 0 and m % i == 0):
            i += 1
        return i

    b_options = upper_limit - 1
    total_solutions = 0
    answer_list = []
    for a in range(2, upper_limit + 1):
        answers = 0
        for b in range(2, upper_limit + 1):
            if a ** b not in answer_list:
                answers += 1
                answer_list.append(a**b)
            # else:
            #     print
        if mathf.is_prime(a):
            total_solutions += b_options
            # print(f"{a}: {b_options} (real: {answers})")
            continue
        single_factor = get_single_factor(a)
        if single_factor == 0:
            total_solutions += b_options
            # print(f"{a}: {b_options} (real: {answers})")
            continue
        else:
            # a can be written as x^power
            power = int(math.log(a) / math.log(single_factor))

            prime_power = mathf.is_prime(power)

            calculated_by_x = upper_limit // power

            lower_bound = max(calculated_by_x, power - 1)

            solutions = b_options - lower_bound + 1

            print_statement = ""

            if power > 2:
                for b in range(lower_bound + 1, upper_limit + 1):
                    if mathf.is_prime(b) and prime_power:
                        print_statement += f"{b} and {power} are prime \n"
                        continue
                    for previous_power in range(2, power):
                        if b % previous_power == 0:
                            if (b // previous_power) * power <= upper_limit:
                                print_statement += f"{b} // {previous_power} * {power} <= {upper_limit} \n"
                                solutions -= 1
                                break
                        if power % previous_power == 0:
                            if (power // previous_power) * b <= upper_limit:
                                print_statement += f"divisible power: {power} // {previous_power} * {b} <= {upper_limit} \n"
                                solutions -= 1
                                break
            # first_shared_power = 2 * power
            #
            # # The first is always already calculated.
            # total_calculated = 1
            # print_statement = ""
            # for previous_power in range(1, power):
            #     # smallest_factor = smallest_common_factor(power, previous_power)
            #     # if first_shared_power % smallest_factor != 0:
            #     #     print_statement += f"not a multiple of factor: {first_shared_power}, {smallest_factor} \n"
            #     #     first_shared_power -= first_shared_power % smallest_factor
            #     #     first_shared_power += smallest_factor
            #
            #     first_shared_power_index =  first_shared_power // previous_power
            #     if first_shared_power_index > upper_limit * previous_power:
            #         # This power is so large, this previous power has not calculated it yet.
            #         continue
            #
            #     remaining_numbers = upper_limit - first_shared_power_index
            #
            #     print_statement += f"{remaining_numbers} = {upper_limit} - {first_shared_power_index}\n"
            #
            #     divisor = power
            #
            #     if previous_power != 1:
            #         if power % previous_power == 0:
            #             divisor = previous_power
            #
            #     already_calculated = math.ceil(remaining_numbers / divisor)
            #
            #     print_statement += f"{already_calculated} = {remaining_numbers} // {divisor}\n"
            #
            #     total_calculated += already_calculated
            #
            #     if previous_power != power - 1:
            #         first_shared_power += power * already_calculated
            #     print_statement += f"{power}, {previous_power}: {already_calculated= }, {first_shared_power= }, {total_calculated= } \n"
            #     # # already_reached = power * (previously_calculated + 2)
            #     # remaining_options = b_options - 2 * (previous_power - 1)
            #     # newly_calculated = math.ceil(remaining_options // previous_power)
            #     # # if power % previous_power == 0:
            #     # #     newly_calculated *= 2
            #     # previously_calculated += newly_calculated
            #     # print(power, previous_power, previously_calculated, remaining_options, newly_calculated)

            # already_calculated += upper_limit // power

            # if solutions != answers:
            #     print(print_statement)
            #     print(f"{a}: {solutions} (real: {answers}) {upper_limit= }")
            #     return None
            total_solutions += solutions
    # print(total_solutions, len(answer_list))
    # print(sorted(answer_list))
    return total_solutions

In [211]:
p.test_once("counting_solutions")

9187 found after a separate test in 257.071900 ms by counting_solutions (first)


In [5]:
@solution(P029, best=True, make_fast=True)
def best_solve():
    return brute_force()

In [6]:
p.test_all()

First result: 1 found in  0.000005 seconds.
Best result: 1 found in  0.000004 seconds.
